In [1]:
# !pip install langchain langchain-text-splitters langchain-openai langchain-chroma chromadb python-dotenv

import os
from dotenv import load_dotenv

load_dotenv()

openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_base_url or not openrouter_api_key:
    raise ValueError("OPENROUTER_BASE_URL or OPENROUTER_API_KEY is not set")

## 1. Вхідні фінансові дані

In [2]:
raw_financial_data = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

## 2. Чанкування тексту - розбиття по компаніях за допомогою `RecursiveCharacterTextSplitter`

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=250,
    chunk_overlap=0
)

docs = text_splitter.create_documents([raw_financial_data])

print(f"Кількість створених чанків: {len(docs)}")
for i, doc in enumerate(docs, 1):
    print(f"\n--- Чанк {i} ---")
    print(doc.page_content)

Кількість створених чанків: 4

--- Чанк 1 ---
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

--- Чанк 2 ---


ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

--- Чанк 3 ---


РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

--- Чанк 4 ---
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від

## 3. Сховище Chroma та налаштування Retriever
Індексуємо документи у векторну базу Chroma з назвою колекції `finansovi_zvity_kompaniy` та налаштовуємо retriever на пошук `k=2` релевантних документів.

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",  # https://openrouter.ai/openai/text-embedding-3-small
    api_key=openrouter_api_key,
    base_url=openrouter_base_url
)

collection_name = "finansovi_zvity_kompaniy"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name=collection_name
)

k = 2
retriever = vectorstore.as_retriever(search_kwargs={"k": k})

print(f"Векторне сховище '{collection_name}' успішно створено. Retriever налаштовано з k={k}.")

Векторне сховище 'finansovi_zvity_kompaniy' успішно створено. Retriever налаштовано з k=2.


## 4. Створення RAG ланцюга
Налаштовуємо мовну модель `ChatOpenAI`, промпт та RAG ланцюг за допомогою LCEL.

#### З конспекту:
1. `{"context": retriever | format_docs, "question": RunnablePassthrough()}`: Ця частина працює паралельно. Вона надсилає запитання користувача до ретривера для отримання документів, які потім форматуються в один рядок функцією format_docs. Одночасно RunnablePassthrough передає оригінальне запитання без змін.

2. `| prompt`: Контекст і запитання подаються в наш шаблон промпта.

3. `| llm` : Відформатований промпт надсилається до LLM.

4. `| StrOutputParser()`: Це очищує вивід LLM до простого рядка.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  #https://openrouter.ai/openai/gpt-4o-mini
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    temperature=0
)

system_prompt = """Ти — фінансовий аналітик-помічник. Твоє завдання — відповідати на запитання щодо фінансових показників компаній, використовуючи виключно наданий контекст.

Вимоги:
1. Використовуй тільки інформацію з наданого контексту.
2. Давай точні та лаконічні відповіді з конкретними цифрами та фактами.
3. Якщо в наданому контексті немає інформації для відповіді на запитання (наприклад, про запитувану компанію або показник нічого не сказано), чітко зазнач, що на основі наданого контексту інформація відсутня.
4. Відповідай українською мовою.

Контекст:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])


def format_docs(docs):
    """Допоміжна функція для форматування отриманих документів"""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

## 5. Тестування RAG системи
Тестові питання, форматовані відповіді.

In [6]:
from datetime import datetime

questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?"
]

print(f"{"=" * 30}{datetime.now():%Y-%m-%d %H:%M:%S}{"=" * 30}")
for i, question in enumerate(questions, 1):
    print(f"\nПитання {i}: {question}")
    answer = rag_chain.invoke(question)
    print(f"Відповідь: {answer}")
    print("-" * 70)

# Очікуваний результат
#
# Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
# Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн доларів.
# ----------------------------------------------------------------------
#
# Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
# Відповідь: 5100 осіб.
# ----------------------------------------------------------------------
#
# Питання 3: Який показник ROE у РітейлМакс Груп?
# Відповідь: Показник ROE у РітейлМакс Груп становить 9.3 відсотка.
# ----------------------------------------------------------------------
#
# Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
# Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
# ----------------------------------------------------------------------
#
# Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
# Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
# ----------------------------------------------------------------------
#
# Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
# Відповідь: На основі наданого контексту, інформація про чистий прибуток ЕкоПакування Солюшнс відсутня.
# ----------------------------------------------------------------------


==============================2026-11-05 17:04:34==============================

Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------

Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
-------------------------------------------------------